# Europe's Airbnb rules, one ruler for ten cities

Every big European city regulates Airbnb its own way. Amsterdam allows an entire home to be
rented to tourists 30 nights a year; Paris, London, Vienna and Berlin 90; Barcelona is ending its
tourist-flat licences in 2028; Lisbon, Rome, Brussels and Prague have no night cap at all.
This notebook measures **every Airbnb listing in ten cities the same way** (Inside Airbnb,
September 2026 snapshots, 299,434 listings) and then checks each city against its own rule.

**The ruler**

* **Entire homes only** for nights and hosts: that's where the rules bite.
* **Nights booked** = Inside Airbnb's own estimate for the last 12 months
  (`estimated_occupancy_l365d`): reviews ÷ a 50% review rate × the average stay (or the minimum
  stay if longer), capped at 70% of the year. Next to it, a **careful** version: 70% review
  rate and 2-night stays, which gives much lower numbers.
* **Short-stay homes** = entire homes that take stays under 31 nights. The night caps only apply
  to those; homes that only take 31+ nights are counted separately.
* **Active** = at least one review in the last 12 months.

Nights are an estimate from reviews, not real bookings. And going over a cap isn't always
illegal: permits and planning permission aren't in the data.

**Privacy:** only counts and shares come out of this notebook. No host names, listing names,
addresses or registration numbers are ever shown; the licence field is only sorted into classes.

In [ ]:
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

DATA = Path("data")
CITIES = ["amsterdam", "paris", "london", "vienna", "berlin", "barcelona", "lisbon", "rome",
          "brussels", "prague"]
NAMES = {c: c.title() for c in CITIES}
# The cap on nights an entire home may be rented to tourists per year (None = no night cap)
CAP = {"amsterdam": 30, "paris": 90, "london": 90, "vienna": 90, "berlin": 90,
       "barcelona": None, "lisbon": None, "rome": None, "brussels": None, "prague": None}
MONTHLY = 31                     # minimum stay that takes a listing out of the tourist rules
REVIEW_RATE, MAX_OCC = 0.5, 0.70
CAREFUL = dict(review_rate=0.7, avg_stay=2)
# Inside Airbnb shows every price with a "$", but it's the local currency.
# Euros per unit, ECB reference rates around the snapshot date.
EUR_PER = {"london": 1 / 0.86, "prague": 1 / 25.0}

plt.rcParams.update({"figure.figsize": (10, 4.5), "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.3})
BLUE, ORANGE = "#4F79B8", "#E8641D"
pd.set_option("display.float_format", "{:.3f}".format)

def hbar(series, title, fmt="{:.0%}", hot=None, xmax=None, labels=None):
    """Horizontal bars, biggest first at the top. hot = labels to colour orange."""
    hot = [hot] if isinstance(hot, str) else (hot or [])
    names, values = list(series.index)[::-1], [float(v) for v in series.values][::-1]
    xmax = xmax or max(values) * 1.18
    fig, ax = plt.subplots(figsize=(10, 0.45 * len(values) + 1.2))
    ax.barh([str(n) for n in names], values, color=[ORANGE if n in hot else BLUE for n in names])
    ax.set(title=title, xlim=(0, xmax))
    if "%" in fmt:
        ax.xaxis.set_major_formatter(lambda v, _: f"{v:.0%}" if xmax > 0.1 else f"{v:.1%}")
    texts = (labels[::-1] if labels is not None else [fmt.format(v) for v in values])
    for i, (v, t) in enumerate(zip(values, texts)):
        ax.text(v + xmax * 0.01, i, t, va="center")
    plt.show()

## 1. Download

The detailed listings file for each city from [Inside Airbnb](https://insideairbnb.com)
(CC BY 4.0), September 2026 snapshots, about 170 MB in total. Files that are already there are
skipped.

Inside Airbnb keeps the latest snapshots on its download page; older ones move to an archive
you have to ask for. If a link below has stopped working, use the newest snapshot for that city:
the numbers will shift a little, the method stays the same.

In [ ]:
SNAPSHOTS = {
    "amsterdam": "the-netherlands/north-holland/amsterdam/2026-09-16",
    "paris": "france/ile-de-france/paris/2026-09-16",
    "london": "united-kingdom/england/london/2026-09-17",
    "barcelona": "spain/catalonia/barcelona/2026-09-17",
    "berlin": "germany/be/berlin/2026-09-19",
    "lisbon": "portugal/lisbon/lisbon/2026-09-19",
    "rome": "italy/lazio/rome/2026-09-17",
    "vienna": "austria/vienna/vienna/2026-09-17",
    "brussels": "belgium/bru/brussels/2026-09-20",
    "prague": "czech-republic/prague/prague/2026-09-20",
}


def download(city, path):
    out = DATA / city / "listings.csv.gz"
    if out.exists() and out.stat().st_size > 1000:
        return "already there"
    out.parent.mkdir(parents=True, exist_ok=True)
    url = f"https://data.insideairbnb.com/{path}/data/listings.csv.gz"
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        tmp = out.with_suffix(".part")
        with open(tmp, "wb") as f:
            for chunk in r.iter_content(1 << 20):
                f.write(chunk)
        tmp.replace(out)
    return f"{out.stat().st_size / 1e6:.0f} MB"


for city, path in SNAPSHOTS.items():
    try:
        print(f"{city:10s} {download(city, path)}")
    except Exception as e:
        print(f"{city:10s} FAILED: {e}")

## 2. Prepare: one table, one ruler

Every listing gets the same columns: entire home or not, short-stay or monthly, active or not,
nights (both estimates), estimated earnings in euros, the licence class, and how many entire
homes its host has (`band`: 1, 2–9 or 10+).

Registration numbers are only checked where Inside Airbnb collects the field. It's empty for
London, Vienna, Brussels and Prague, so nothing is claimed there. Two quirks:

* **Berlin**: about a third of entire homes show Airbnb's host block (name or company and the
  listing address) instead of a number. That's its own class, "name and address only".
* **Barcelona**: Catalan tourist licences (HUTB-…) and Spain's national numbers (ES..TU…
  tourist, ES..NT… non-tourist / seasonal) are told apart. **Paris**: "available with a
  mobility lease only" (bail mobilité) is its own class.

In [ ]:
NUMBER = {   # registration number formats, per city
    "amsterdam": r"\b[0-9A-Z]{4}\s?[0-9A-Z]{4}\s?[0-9A-Z]{4}\s?[0-9A-Z]{4}\s?[0-9A-Z]{4}\b",
    "paris": r"\b\d{13}\b",
    "berlin": r"\b\d{2}/Z/[A-Z]{2}/\d{6}-\d{2}\b",
    "barcelona": r"HUTB-?\d+|ES[A-Z]{2}(?:TU|NT)\d",
    "lisbon": r"\b\d{1,7}(?:/AL)?\b",
    "rome": r"\bIT\d{6}[A-Z]\d[A-Z0-9]{8}\b",
}
HAS_FIELD = list(NUMBER)


def estimate_nights(df, review_rate=REVIEW_RATE, avg_stay=3.0):
    stay = df["minimum_nights"].clip(lower=avg_stay)
    return (df["number_of_reviews_ltm"].fillna(0) / review_rate * stay).clip(upper=int(365 * MAX_OCC))


def licence_class(city, s):
    """Sort what a listing shows in its licence field into a few classes (the text is never kept)."""
    s = s.fillna("").astype(str).str.strip()
    up = s.str.upper()
    cls = pd.Series("none", index=s.index)
    cls[s != ""] = "other text"
    cls[up.str.contains("EXEMPT")] = "exempt"
    cls[up.str.contains("PENDING|APPLIED|IN PROCESS|HAVEN")] = "applied"
    cls[up.str.contains("MOBILIT")] = "mobility lease"
    cls[up.str.contains("NAME") & up.str.contains("ADDRESS")] = "name and address only"
    if city in NUMBER:
        cls[up.str.contains(NUMBER[city], regex=True) & ~cls.isin(["mobility lease"])] = "number"
    if city == "barcelona":
        tourist = up.str.contains(r"HUTB|ES[A-Z]{2}TU\d", regex=True)
        cls[up.str.contains(r"ES[A-Z]{2}NT\d", regex=True) & ~tourist] = "non-tourist number"
    return cls


def load(city):
    df = pd.read_csv(DATA / city / "listings.csv.gz", low_memory=False)
    df["city"] = city
    df["entire"] = df["room_type"] == "Entire home/apt"
    df["monthly"] = df["minimum_nights"] >= MONTHLY
    df["active"] = df["number_of_reviews_ltm"].fillna(0) > 0
    df["nights"] = df["estimated_occupancy_l365d"].fillna(0)
    df["nights_careful"] = estimate_nights(df, **CAREFUL)
    df["revenue"] = df["estimated_revenue_l365d"].fillna(0) * EUR_PER.get(city, 1)
    df["lic"] = licence_class(city, df["license"])
    df["barcelona_hutb"] = df["license"].fillna("").astype(str).str.upper().str.contains("HUTB")
    df["host_entire"] = df["calculated_host_listings_count_entire_homes"].fillna(0)
    df["price_eur"] = pd.to_numeric(df["price"].astype(str).str.replace(r"[$,]", "", regex=True),
                                    errors="coerce") * EUR_PER.get(city, 1)
    df["band"] = np.select([df.host_entire <= 1, df.host_entire < 10], ["1", "2-9"], "10+")
    df = df.drop(columns=["license"])          # never needed again
    return df


A = pd.concat([load(c) for c in CITIES], ignore_index=True)
E = A[A.entire]                 # entire homes
S = E[~E.monthly]               # short-stay entire homes, where the night caps apply


def by_city(df, f):
    return pd.DataFrame([{"city": NAMES[c], **f(df[df.city == c])} for c in CITIES]).set_index("city")


print(f"{len(A):,} listings, {len(E):,} entire homes, {len(S):,} short-stay entire homes")
by_city(A, lambda d: dict(snapshot=pd.to_datetime(d.last_scraped).max().date(), listings=len(d),
                          share_entire=d.entire.mean(), hosts=d.host_id.nunique(),
                          licence_field=d.city.iloc[0] in HAS_FIELD))

## Finding 1: half of them are ghosts

A listing that got no review in a year probably had no guests we can see. Counting listings
makes some cities look a lot busier than they are.

In [ ]:
ghosts = by_city(A, lambda d: dict(
    listings=len(d), no_guest_12m=1 - d.active.mean(),
    never_reviewed=(d.number_of_reviews.fillna(0) == 0).mean(),
    entire_first_guest_since_2024=(pd.to_datetime(d[d.entire].first_review, errors="coerce")
                                   >= "2024-01-01").mean()))
print(f"All ten cities: {1 - A.active.mean():.0%} of listings had no review in 12 months")
hbar(ghosts.no_guest_12m.sort_values(ascending=False), "Listings with no review in the last 12 months",
     hot=["London", "Paris"])
ghosts

## Finding 2: the typical home, and a stress test

Median nights booked in the last 12 months, active entire homes. Then the careful estimate: if
Amsterdam is still last with numbers built to come out low, the gap isn't a quirk of the model.

In [ ]:
NIGHT_LINES = [30, 60, 90, 120, 180]


def nights_row(d):
    act = d[d.active]
    r = dict(homes=len(d), active=len(act), median_nights=act.nights.median(),
             median_nights_careful=act.nights_careful.median())
    for n in NIGHT_LINES:
        r[f"over{n}"] = (d.nights > n).mean()
    return r


nights = by_city(E, nights_row)
hbar(nights.median_nights.sort_values(ascending=False),
     "Typical (median) nights booked a year, active entire homes", fmt="{:.0f}", hot="Amsterdam")
hbar(nights.median_nights_careful.sort_values(ascending=False),
     "The same with the careful estimate (70% review rate, 2-night stays)", fmt="{:.0f}", hot="Amsterdam")
nights

## Finding 3: over their own cap

Short-stay entire homes in the five cities with a night cap, against that city's own cap.

In [ ]:
capped = [c for c in CITIES if CAP[c]]
own = pd.DataFrame([{
    "city": NAMES[c], "cap": CAP[c], "homes": len(d := S[S.city == c]),
    "share_over_cap": (d.nights > CAP[c]).mean(),
    "share_over_cap_careful": (d.nights_careful > CAP[c]).mean(),
    "share_over_2x_cap": (d.nights > 2 * CAP[c]).mean(),
    "active_share_over_cap": (d[d.active].nights > CAP[c]).mean()} for c in capped]).set_index("city")
hbar(own.share_over_cap.sort_values(ascending=False),
     "Short-stay entire homes booked more nights than their city's cap", hot="Berlin")
own

## Finding 4: locals or pros?

The same test by how many entire homes the host has. Hosts with one home go over the cap far
less often than hosts with several.

In [ ]:
by_host = pd.DataFrame([{"city": NAMES[c], "band": b, "homes": len(x),
                         "share_over_cap": (x.nights > CAP[c]).mean(),
                         "share_over_cap_careful": (x.nights_careful > CAP[c]).mean()}
                        for c in capped for b, x in S[S.city == c].groupby("band")])
t = by_host.pivot(index="city", columns="band", values="share_over_cap")[["1", "2-9", "10+"]]
ax = t.plot.bar(rot=0, color=[BLUE, "#8FA8D0", ORANGE],
                title="Short-stay homes over the cap, by how many entire homes the host has")
ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
plt.show()
by_host

## Finding 5: follow the money

Estimated earnings in the last 12 months (Inside Airbnb's `estimated_revenue_l365d`, converted to
euros). Almost all of it is made past the cap.

In [ ]:
money = pd.DataFrame([{
    "city": NAMES[c], "cap": CAP[c],
    "revenue_over_cap_eur": (d := S[S.city == c]).revenue[d.nights > CAP[c]].sum(),
    "share_revenue_over_cap": d.revenue[d.nights > CAP[c]].sum() / max(d.revenue.sum(), 1),
    "median_revenue_over_cap": d.revenue[d.nights > CAP[c]].median()} for c in capped]).set_index("city")
hbar(money.share_revenue_over_cap.sort_values(ascending=False),
     "Share of short-stay earnings made by homes over the cap", hot="Berlin")
money

## Finding 6: who runs them?

Hosts with 10 or more entire homes, and the very biggest hosts. Counts only: no host is named.
Most big "hosts" are management companies, so "runs", not "owns".

In [ ]:
def hosts_row(d):
    big = d.host_entire >= 10
    hc = d.groupby("host_id").size().sort_values(ascending=False)
    return dict(homes=len(d), share_homes_host10plus=big.mean(),
                share_homes_single=(d.host_entire <= 1).mean(),
                share_revenue_host10plus=d.revenue[big].sum() / max(d.revenue.sum(), 1),
                biggest_host_homes=hc.iloc[0], hosts_50plus=(hc >= 50).sum(),
                share_homes_biggest10=hc.iloc[:10].sum() / len(d))


hosts = by_city(E, hosts_row)
hbar(hosts.share_homes_host10plus.sort_values(ascending=False),
     "Entire homes run by a host with 10+ entire homes", hot="Barcelona")
hosts

## Finding 7: where they are

Every city on the same grid of 1 × 1 km squares (a flat approximation around the city's
mean position, accurate enough at this scale). How many squares hold half of a city's entire homes?

In [ ]:
grid = []
for c in CITIES:
    d = E[E.city == c]
    lat0, lon0 = d.latitude.mean(), d.longitude.mean()
    gx = np.floor((d.longitude - lon0) * 111.32 * np.cos(np.radians(lat0))).astype(int)
    gy = np.floor((d.latitude - lat0) * 111.32).astype(int)
    grid.append(pd.DataFrame({"gx": gx, "gy": gy}).value_counts().rename("homes").reset_index()
                .assign(city=NAMES[c]))
grid = pd.concat(grid)
hot = grid.groupby("city", sort=False).homes.agg(
    squares="size", busiest_square="max",
    top10_share=lambda h: h.nlargest(10).sum() / h.sum(),
    squares_for_half=lambda h: int((h.sort_values(ascending=False).cumsum() < h.sum() / 2).sum() + 1))
hbar(hot.busiest_square.sort_values(ascending=False), "Entire homes in the busiest 1 km² of each city",
     fmt="{:,.0f}", hot="Paris")
hot

## Finding 8: the registration number

What the licence field shows, in the six cities where Inside Airbnb collects it, for active
short-stay entire homes. "Number" only means the right format; nobody can check from here
whether a number is real.

In [ ]:
reg = []
for c in HAS_FIELD:
    d = S[(S.city == c) & S.active]
    reg.append({"city": NAMES[c], "homes": len(d), **d.lic.value_counts(normalize=True).to_dict()})
reg = pd.DataFrame(reg).set_index("city").fillna(0)
reg

## Finding 9: the 31-night trick

A home that only takes stays of 31 nights or more is out of the tourist rules. Where short
stays get hard, that's what grows: Berlin and Barcelona. In Paris, the "mobility lease"
(bail mobilité, 1 to 10 months for students, trainees and people on a work assignment) does the same job.

In [ ]:
monthly = by_city(E, lambda d: dict(homes=len(d), min_nights_31plus=d.monthly.mean(),
                                    min_nights_90plus=(d.minimum_nights >= 90).mean(),
                                    monthly_and_active=(d.monthly & d.active).mean()))
hbar(monthly.min_nights_31plus.sort_values(ascending=False),
     "Entire homes that only take stays of 31 nights or more", hot=["Barcelona", "Berlin"])

p = E[E.city == "paris"]
mob = p.lic == "mobility lease"
print(f"Paris: {mob.sum():,} entire homes on a mobility lease only ({mob.mean():.0%}); "
      f"{(p[mob].minimum_nights >= 30).mean():.0%} of them take 30+ nights minimum")
monthly

## Finding 10: Berlin and Barcelona up close

Berlin: registration numbers vs the host block (name and address) typed into the field.
Barcelona: tourist licences, which won't be renewed after November 2028, vs non-tourist
(seasonal, 31+ night) numbers vs nothing.

In [ ]:
berlin = (A[A.city == "berlin"].groupby(["room_type", "lic"])
          .agg(listings=("id", "size"), active=("active", "mean"), monthly=("monthly", "mean"),
               median_nights=("nights", "median")))
display(berlin)

b = A[A.city == "barcelona"].copy()
b["kind"] = np.select([b.lic == "non-tourist number", b.barcelona_hutb | (b.lic == "number"), b.lic == "none"],
                      ["non-tourist (31+ nights)", "tourist licence", "no number"], "other")
b.groupby(["room_type", "kind"]).agg(listings=("id", "size"), active=("active", "mean"),
                                     median_min_nights=("minimum_nights", "median"),
                                     median_nights=("nights", "median"))

## Finding 11: the guest's view

Asking price per night (active entire homes, on the day the data was collected) and the share
of homes rated 4.9 or higher (5+ reviews), single-home hosts vs hosts with 10+ homes.

In [ ]:
def guest_row(d):
    r = dict(median_price=d[d.active].price_eur.median())
    for band in ["1", "10+"]:
        x = d[d.band == band]
        rated = x[x.number_of_reviews >= 5]
        r[f"price_{band}"] = x[x.active].price_eur.median()
        r[f"rated_4.9_{band}"] = (rated.review_scores_rating >= 4.9).mean()
    return r


guest = by_city(E, guest_row)
t = guest[["rated_4.9_1", "rated_4.9_10+"]].rename(columns={"rated_4.9_1": "host with 1 home",
                                                            "rated_4.9_10+": "host with 10+ homes"})
ax = t.plot.bar(rot=30, color=[BLUE, ORANGE], title="Entire homes rated 4.9 or higher (5+ reviews)")
ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
plt.show()
guest

## Summary

| City | Cap (nights) | Median nights | Short-stay homes over own cap | Earnings made over the cap | Homes run by 10+ hosts | No review in 12 months |
|---|---|---|---|---|---|---|
| Amsterdam | 30 | **31** | 32% | 88% | 1% | 38% |
| Paris | 90 | 94 | 31% | 85% | 20% | 47% |
| London | 90 | 60 | 21% | 75% | 29% | **49%** |
| Vienna | 90 | 79 | 31% | 79% | 38% | 35% |
| Berlin | 90 | **176** | **52%** | **91%** | 18% | 44% |
| Barcelona | none | 162 | | | **56%** | 35% |
| Lisbon | none | 84 | | | 31% | 24% |
| Rome | none | 96 | | | 18% | 26% |
| Brussels | none | 126 | | | 12% | 24% |
| Prague | none | 120 | | | 44% | 20% |

Amsterdam is the odd one out: the only city where the typical home is rented out about as many
nights as the cap allows, and it stays last with the careful estimate. Everywhere else a big
share of homes goes past its own city's cap, and that's where most of the money is made. Where
short stays get hard, the month-long let takes over.

**Limits:** nights and earnings are estimated from reviews, not real bookings. Prices are the
asking price on the day the data was collected. One snapshot per city. Permits, planning
permission and second-home rules aren't in the data, so "over the cap" doesn't always mean
breaking the rules. Registration numbers are only checked in six cities, and only for the right
format. Airbnb only. Ten cities show a pattern, not proof of what caused it.